## Interactive per-unit comparison
System bases assume UDC = 100 V, R_L = 0.25 Ω, and L = 1.6 mH. Adjust the dials for M1, M2, phase angle θ (V2 relative to V1), inductance L, resistance R_L, and frequency to explore the waveforms, phasors, and a power triangle scaled to the system's maximum active/reactive power.

In [10]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io
from matplotlib.ticker import FuncFormatter
from ipywidgets import FloatSlider, HBox, VBox, Layout, HTML, Image, Label
from IPython.display import display
plt.ioff()


UDC = 100.0          # volts
R_SYSTEM = 0.25      # ohms
L_SYSTEM = 1.6e-3    # henrys
FREQ_MIN = 20.0      # hertz
FREQ_MAX = 500.0
M_MIN = 0.0
M_MAX = 1.2

V_BASE = UDC
I_BASE = V_BASE / R_SYSTEM if R_SYSTEM > 1e-9 else 10.0

fig = plt.figure(figsize=(16, 4))
axes = fig.subplots(1, 3)
wave_ax, vector_ax, power_ax = axes
figure_image = Image(layout=Layout(width="100%"))
summary_html = HTML()




power_ax.set_xlabel('P (kW)')
power_ax.set_ylabel('Q (kVAr)')
power_ax.xaxis.set_major_formatter(FuncFormatter(lambda val, _: f'{val:.1f}'))
power_ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f'{val:.1f}'))







def estimate_power_span():
    m_vals = np.linspace(M_MIN, M_MAX, 9)
    theta_vals = np.linspace(-180.0, 180.0, 61)
    freq_vals = [FREQ_MIN, FREQ_MAX]

    max_p = 0.0
    max_q = 0.0
    for m1 in m_vals:
        for m2 in m_vals:
            for theta in theta_vals:
                theta_rad = np.deg2rad(theta)
                v1 = np.array([m1, 0.0])
                v2 = m2 * np.array([np.cos(theta_rad), np.sin(theta_rad)])
                diff = v2 - v1
                diff_real_peak = diff[0] * V_BASE
                diff_imag_peak = diff[1] * V_BASE

                v_r_rms = abs(diff_real_peak) / np.sqrt(2)
                v_l_rms = abs(diff_imag_peak) / np.sqrt(2)

                for freq in freq_vals:
                    omega = 2 * np.pi * freq
                    if R_SYSTEM > 1e-9:
                        p = (v_r_rms ** 2) / R_SYSTEM
                        max_p = max(max_p, p)
                    if L_SYSTEM > 1e-9 and omega > 1e-9:
                        q = (v_l_rms ** 2) / (omega * L_SYSTEM)
                        max_q = max(max_q, q)
    span = np.hypot(max_p, max_q)
    return max(max_p, max_q, span) * 1.1


POWER_SPAN = estimate_power_span()


def update_plot(theta_deg, inductance_h, resistance_ohm, m1_pu, m2_pu, frequency_hz):
    theta_rad = np.deg2rad(theta_deg)
    omega = 2 * np.pi * frequency_hz

    period = 1.0 / max(frequency_hz, 1e-6)
    t = np.linspace(0, period, 400)

    v1_wave_pu = m1_pu * np.sin(omega * t)
    v2_wave_pu = m2_pu * np.sin(omega * t + theta_rad)
    diff_wave_pu = v2_wave_pu - v1_wave_pu

    v1_vec_pu = np.array([m1_pu, 0.0])
    v2_vec_pu = m2_pu * np.array([np.cos(theta_rad), np.sin(theta_rad)])
    diff_vec_pu = v2_vec_pu - v1_vec_pu
    diff_mag_pu = np.hypot(*diff_vec_pu)

    wave_ax.clear()
    wave_ax.plot(t, v1_wave_pu, label=r'$V_1(t)$', color='tab:orange')
    wave_ax.plot(t, v2_wave_pu, label=r'$V_2(t)$', color='tab:blue')
    wave_ax.plot(t, diff_wave_pu, label='V2 - V1', linestyle='--', color='tab:green')
    wave_ax.set_title('Per-unit waveforms')
    wave_ax.set_xlabel('time (s)')
    wave_ax.set_ylabel('per-unit (V/UDC)')
    wave_ax.grid(True, alpha=0.3)
    wave_ax.legend(loc='upper right')

    vector_ax.clear()
    vector_ax.set_aspect('equal', 'box')

    diff_real_peak = diff_vec_pu[0] * V_BASE
    diff_imag_peak = diff_vec_pu[1] * V_BASE

    if resistance_ohm > 1e-9:
        v_r_rms = diff_real_peak / np.sqrt(2)
        i_real_peak = diff_real_peak / resistance_ohm
        p_abs = (abs(v_r_rms) ** 2) / resistance_ohm
        p_actual = np.sign(diff_real_peak) * p_abs
    else:
        v_r_rms = 0.0
        i_real_peak = 0.0
        p_actual = 0.0

    if inductance_h > 1e-9 and omega > 1e-9:
        v_l_rms = diff_imag_peak / np.sqrt(2)
        i_imag_peak = -diff_imag_peak / (omega * inductance_h)
        q_abs = (abs(v_l_rms) ** 2) / (omega * inductance_h)
        sign_q = np.sign(diff_imag_peak)
        q_actual = sign_q * q_abs
    else:
        i_imag_peak = 0.0
        q_actual = 0.0

    current_vec_pu = np.array([
        i_real_peak / I_BASE,
        i_imag_peak / I_BASE,
    ])
    current_mag_pu = np.hypot(*current_vec_pu)
    current_mag_actual = current_mag_pu * I_BASE
    current_angle_deg = (
        np.degrees(np.arctan2(current_vec_pu[1], current_vec_pu[0]))
        if current_mag_pu > 1e-8 else 0.0
    )

    span = max(
        1.2,
        m1_pu * 1.3,
        m2_pu * 1.3,
        diff_mag_pu * 1.5,
        current_mag_pu * 1.6,
    )
    vector_ax.set_xlim(-span, span)
    vector_ax.set_ylim(-span, span)
    vector_ax.axhline(0, color='0.8', linewidth=1)
    vector_ax.axvline(0, color='0.8', linewidth=1)

    arrow_width = max(0.004 * span, 0.002)
    head_width = max(0.04 * span, 0.03)

    vector_ax.arrow(0, 0, *v1_vec_pu, color='tab:orange', width=arrow_width,
                    length_includes_head=True, head_width=head_width)
    vector_ax.arrow(0, 0, *v2_vec_pu, color='tab:blue', width=arrow_width,
                    length_includes_head=True, head_width=head_width)
    vector_ax.arrow(*v1_vec_pu, *diff_vec_pu, color='tab:green', width=arrow_width,
                    length_includes_head=True, head_width=head_width)
    vector_ax.arrow(0, 0, *current_vec_pu, color='tab:red', width=arrow_width,
                    length_includes_head=True, head_width=head_width)

    if np.hypot(*v1_vec_pu) > 1e-6:
        vector_ax.text(*(v1_vec_pu * 1.05), 'V1', color='tab:orange')
    if np.hypot(*v2_vec_pu) > 1e-6:
        vector_ax.text(*(v2_vec_pu * 1.05), 'V2', color='tab:blue')
    if diff_mag_pu > 1e-6:
        vector_ax.text(*(v1_vec_pu + diff_vec_pu * 0.5), 'V2 - V1', color='tab:green')
    if current_mag_pu > 1e-4:
        vector_ax.text(*(current_vec_pu * 1.05), 'I', color='tab:red')

    phase_delta = ((theta_deg + 180) % 360) - 180
    diff_mag_actual = diff_mag_pu * V_BASE

    info_lines = [
        f'M1 = {m1_pu:.2f} p.u. ({m1_pu * V_BASE:.1f} V)',
        f'M2 = {m2_pu:.2f} p.u. ({m2_pu * V_BASE:.1f} V)',
        f'θ = {theta_deg:.1f}°',
        f'|V2 - V1| = {diff_mag_pu:.3f} p.u. ({diff_mag_actual:.1f} V)',
        f'|I| = {current_mag_pu:.3f} p.u. ({current_mag_actual:.3f} A)',
        f'∠I = {current_angle_deg:.1f}°',
        f'L = {inductance_h * 1e3:.2f} mH, R_L = {resistance_ohm:.3f} Ω',
        f'Phase delta (V2 - V1) = {phase_delta:.1f}°',
    ]
    summary_html.value = '<br>'.join(info_lines)
    circle = plt.Circle((0, 0), 1.0, edgecolor='0.5', facecolor='none', linestyle=':')
    vector_ax.add_patch(circle)
    vector_ax.set_title('Phasor diagram (per-unit)')

    power_ax.clear()
    power_ax.xaxis.set_major_formatter(FuncFormatter(lambda val, _: f'{val:.1f}'))
    power_ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f'{val:.1f}'))
    power_ax.axhline(0, color='0.7', linewidth=1, linestyle=':')
    power_ax.axvline(0, color='0.7', linewidth=1, linestyle=':')

    power_span_kw = 15.0
    p_kw = p_actual / 1e3
    q_kvar = q_actual / 1e3
    s_mag_kw = np.hypot(p_kw, q_kvar)

    power_ax.set_xlim(-power_span_kw, power_span_kw)
    power_ax.set_ylim(-power_span_kw, power_span_kw)
    power_ax.set_aspect('equal', 'box')
    power_ax.set_title('Power triangle')
    power_ax.set_xlabel('P (kW)')
    power_ax.set_ylabel('Q (kVAr)')

    if abs(p_kw) > 1e-6 or abs(q_kvar) > 1e-6:
        triangle_x = [0, p_kw, p_kw]
        triangle_y = [0, 0, q_kvar]
        power_ax.fill(triangle_x, triangle_y, color='tab:purple', alpha=0.1)

    arrow_style = dict(arrowstyle='->', lw=2)
    handles = []
    if abs(p_kw) > 1e-6:
        power_ax.annotate('', xy=(p_kw, 0), xytext=(0, 0),
                          arrowprops=dict(color='tab:orange', **arrow_style))
        handles.append(('P', 'tab:orange'))
    if abs(q_kvar) > 1e-6:
        power_ax.annotate('', xy=(p_kw, q_kvar), xytext=(p_kw, 0),
                          arrowprops=dict(color='tab:blue', **arrow_style))
        handles.append(('Q', 'tab:blue'))
    if s_mag_kw > 1e-6:
        power_ax.annotate('', xy=(p_kw, q_kvar), xytext=(0, 0),
                          arrowprops=dict(color='tab:purple', **arrow_style))
        handles.append(('|S|', 'tab:purple'))
    legend_handles = [mpatches.FancyArrowPatch((0, 0), (0, 0), color=color, arrowstyle='->', lw=2, mutation_scale=10)
                      for label, color in handles]
    for handle, (label, color) in zip(legend_handles, handles):
        handle.set_arrowstyle('->')
        handle.set_linestyle('-')
    if handles:
        power_ax.legend([h for h in legend_handles], [label for label, _ in handles],
                        loc='upper left', fontsize=10)
    footer_lines = [
        f'P = {p_kw:.2f} kW',
        f'Q = {q_kvar:.2f} kVAr',
        f'|S| = {s_mag_kw:.2f} kVA',
    ]
    power_ax.text(0.02, -0.18, ' | '.join(footer_lines), transform=power_ax.transAxes,
                 ha='left', va='top', fontsize=10)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    figure_image.value = buf.read()

def on_slider_change(*_):
    update_plot(
        theta_slider.value,
        L_slider.value,
        R_slider.value,
        m1_slider.value,
        m2_slider.value,
        freq_slider.value,
    )


def build_dial(slider, label, color, unit=None, scale=1):
    slider.description = ''
    slider.orientation = 'vertical'
    slider.layout = Layout(height='180px', width='70px')
    slider.style = {'handle_color': color}

    # readout widget (shows value with units)
    readout = Label()

    if unit:
        def update_readout(change=None):
            readout.value = f"{slider.value * scale:.2f} {unit}"
        slider.observe(update_readout, names='value')
        update_readout()
    else:
        readout.value = f"{slider.value:.2f}"

    return VBox([
        HTML(f'<div style="text-align:center;font-weight:bold">{label}</div>'),
        slider,
        readout
    ])


m1_slider = FloatSlider(value=0.8, min=M_MIN, max=M_MAX, step=0.02)
m2_slider = FloatSlider(value=1.0, min=M_MIN, max=M_MAX, step=0.02)
theta_slider = FloatSlider(value=30.0, min=-180.0, max=180.0, step=1.0)
L_slider = FloatSlider(value=L_SYSTEM, min=0.1e-6, max=1e-3, step=0.1e-6, readout_format = ".6f")
R_slider = FloatSlider(value=R_SYSTEM, min=0.05, max=1.0, step=0.01)
freq_slider = FloatSlider(value=50.0, min=FREQ_MIN, max=FREQ_MAX, step=1.0)

controls = HBox(
    [
    build_dial(m1_slider, 'M1 (p.u.)', '#ff7f0e'),
    build_dial(m2_slider, 'M2 (p.u.)', '#1f77b4'),
    build_dial(theta_slider, 'θ (deg)', '#2ca02c'),
    build_dial(L_slider, 'L', '#9467bd', unit="µH", scale=1e6),   # here’s your µH conversion
    build_dial(R_slider, 'R_L (Ω)', '#e377c2'),
    build_dial(freq_slider, 'f (Hz)', '#8c564b'),
    ],
    layout=Layout(justify_content='space-around'))

display(controls)
display(figure_image)
display(summary_html)

update_plot(theta_slider.value, L_slider.value, R_slider.value, m1_slider.value, m2_slider.value, freq_slider.value)


for slider in (m1_slider, m2_slider, theta_slider, L_slider, R_slider, freq_slider):
    slider.observe(on_slider_change, names='value')


Image(value=b'', layout="Layout(width='100%')")

HTML(value='')

# Active and Reactive Power Equations

We consider two inverters feeding a common RL load through an inductor.  
The differential voltage across the inductor is:

$$
V_L = \sqrt{V_1^2 + V_2^2 - 2 V_1 V_2 \cos\theta}
$$

where:

- \( V_1 = \frac{M_1 U_{DC}}{2} \)  
- \( V_2 = \frac{M_2 U_{DC}}{2} \)  
- \( M_1, M_2 \) = modulation indices (0–1)  
- \( U_{DC} \) = DC link voltage  
- \( \theta \) = phase shift between inverter outputs  

---

## Current in the Inductor
The inductor current magnitude is:

$$
I = \frac{V_L}{Z_{eq}} = \frac{V_L}{\sqrt{R_L^2 + (\omega L)^2}}
$$

where \( R_L \) is the series resistance and \( L \) the inductance.

The **impedance angle** is:

$$
\phi = \tan^{-1}\left(\frac{\omega L}{R_L}\right)
$$

---

## Active Power (P)
The active power delivered by Inverter 1 to the load is:

$$
P = V_1 I \cos(\theta - \phi)
$$

- Depends on modulation \(M_1\), relative angle \(\theta\), and load impedance angle \(\phi\).

---

## Reactive Power (Q)
The reactive power is:

$$
Q = V_1 I \sin(\theta - \phi)
$$

- Positive \(Q\): inductive load behavior  
- Negative \(Q\): capacitive load behavior  

---

## Apparent Power
The complex power is:

$$
S = P + jQ = V_1 I e^{j(\theta - \phi)}
$$

with magnitude:

$$
|S| = V_1 I
$$

---

✅ These equations show how **active and reactive power can be controlled** independently by choosing:
- The ratio \(M_1/M_2\) (which sets \(V_L\)),  
- The phase shift \(\theta\),  
- And knowing the load impedance angle \(\phi\).



## Derivation of \$V\_L\$ using the Law of Cosines

We want the magnitude of the inductor voltage vector:

$$
\vec{V}_L = \vec{V}_1 - \vec{V}_2
$$

---

### 1. Law of cosines

For a triangle with sides \$a, b, c\$ and with angle \$\gamma\$ opposite side \$c\$:

$$
c^2 = a^2 + b^2 - 2ab\cos(\gamma)
$$

---

### 2. Identify the phasor terms

* $$a = |V\_1| = M\_1 \cdot \tfrac{U\_{DC}}{2}$$
* $$b = |V\_2| = M\_2 \cdot \tfrac{U\_{DC}}{2}$$
* $$\gamma = \theta (phase shift between \vec{V}\_1 and \vec{V}\_2$$)
* $$c = |V\_L|$$

---

### 3. Apply the law of cosines

$$
|V_L|^2 = |V_1|^2 + |V_2|^2 - 2\,|V_1|\,|V_2|\,\cos(\theta)
$$

Substitute \$|V\_1|\$ and \$|V\_2|\$:

$$
|V_L|^2 =
\left(M_1 \tfrac{U_{DC}}{2}\right)^2
+
\left(M_2 \tfrac{U_{DC}}{2}\right)^2
-
2\left(M_1 \tfrac{U_{DC}}{2}\right)\left(M_2 \tfrac{U_{DC}}{2}\right)\cos(\theta)
$$

---

### 4. Simplify

$$
|V_L| = \tfrac{U_{DC}}{2}\,\sqrt{\,M_1^2 + M_2^2 - 2M_1M_2\cos(\theta)\,}
$$

If it still doesn’t render, double-check:

* You’re in a **Markdown** cell (use `M` in classic or change type in JupyterLab).
* There are **no backticks** around the text.
* MathJax is enabled (it is by default in Jupyter Notebook/Lab).


# Zero-Sequence Current (ZSC)

## Definition
In a three-phase system, the total current can be decomposed into **positive-sequence**, **negative-sequence**, and **zero-sequence** components.  

- **Zero-sequence current (ZSC)** is when all three phase currents are **equal and in phase**:
  $$
  i_a = i_b = i_c = i_0
  $$

This circulating current does not contribute to useful load power. Instead, it flows through parasitic paths or the DC bus, increasing stress and losses.

---

## Sources of ZSC
Several mechanisms in inverter systems can generate **zero-sequence voltages**, which in turn drive ZSC:

1. **PWM Modulation Strategy**
   - **SVPWM** (Space Vector PWM) introduces inherent zero-sequence voltage injection due to the use of zero vectors.
   - **SPWM** (Sinusoidal PWM) avoids this but is less DC bus–utilization efficient.

2. **Carrier Asynchrony**
   - If two inverters share a DC bus but their PWM carriers are not synchronized, small mismatches create common-mode voltage oscillations, exciting ZSC.

3. **Dead-Time Effects**
   - Dead-time inserted to prevent shoot-through makes the phase output clamp to +½Vdc or –½Vdc depending on current direction.
   - These pulses add a high-frequency zero-sequence voltage.

4. **Device/Parasitic Imbalances**
   - Unequal switching delays, diode recovery, or stray impedances cause asymmetries that generate residual zero-sequence components.

---

## Consequences
- Higher **circulating currents** between inverters.
- Extra **conduction and switching losses**.
- Increased **thermal stress** on devices.
- **Current waveform distortion** and degraded power quality.

---

## Suppression Strategies
To mitigate ZSC:
- Adopt **SPWM** instead of SVPWM.
- Apply **dead-time compensation** (current-direction–based correction).
- Use a **PI controller** on the second inverter to cancel residual ZSC.
- Ensure **carrier synchronization** between inverters.


# Dead-Time Compensation Algorithm

Dead-time is the short interval during which **both switches in a leg are OFF** to prevent shoot-through.  
While necessary for safety, this introduces **voltage distortion** because the phase current freewheels through diodes, clamping the output voltage incorrectly.

---

## Principle of Compensation
- When the **phase current is positive**: during dead-time, the phase voltage is pulled **low**.  
- When the **phase current is negative**: during dead-time, the phase voltage is pulled **high**.  
- This creates a **systematic voltage error** that depends on:
  - The **current direction** (sign of phase current),
  - The **dead-time duration** \(t_{dead}\),
  - The **DC bus voltage** \(V_{dc}\).

---

## Correction Formula
The error voltage introduced by dead-time is approximately:

$$
V_{err} = \frac{V_{dc} \cdot t_{dead}}{T_s}
$$

where:
- \(V_{dc}\) = DC link voltage,  
- \(t_{dead}\) = dead-time duration,  
- \(T_s\) = switching period.  

The **compensated reference** is:

- If \(i_{phase} > 0\):  
  $$
  v_{ref,comp} = v_{ref} + V_{err}
  $$

- If \(i_{phase} < 0\):  
  $$
  v_{ref,comp} = v_{ref} - V_{err}
  $$

---

## Implementation Steps
1. **Measure or estimate phase current polarity**.  
2. **Compute voltage error** \(V_{err}\).  
3. **Adjust the modulation reference** by adding or subtracting \(V_{err}\).  
4. Feed the **compensated reference** into the PWM generator.

---

## Effect
- Cancels the unwanted distortion caused by dead-time.  
- Reduces **zero-sequence voltage and current (ZSC)**.  
- Improves **waveform quality** and lowers harmonic distortion.
